# Notebook 05: Model Comparison --- Discrete+Jumps vs Continuous HMM vs GARCH

Head-to-head comparison of three time-series models on SPY:

| Model | Description | Training |
|-------|-------------|----------|
| **Discrete HMM + Jumps** | Quantile-discretized states, frequency-counted transition matrix, Poisson regime teleportation | Baseline from prior paper |
| **Continuous HMM (Baum-Welch)** | Gaussian emissions, EM-trained transition and emission parameters | New contribution (no jumps) |
| **GARCH(1,1)** | Conditional variance model with MLE-fitted parameters | Classical benchmark |

### Metrics
- **KS pass rate** (%) --- Kolmogorov-Smirnov test at alpha = 0.05
- **Excess kurtosis** --- simulated vs observed
- **ACF-MAE** --- mean absolute error of ACF(|Gt|) over lags 1-252
- **Wasserstein-1 distance** --- mean earth-mover distance
- **Hellinger distance** --- histogram overlap distance

## Setup

In [ ]:
include("../Include.jl");

## Configuration (Tunable)

In [ ]:
# --- TUNABLE PARAMETERS ---
ticker = "SPY";
K_discrete = 13;            # number of discrete states (quantile bins)
K_continuous = 6;           # number of CHMM hidden states
ε = 0.01;                   # discrete jump probability
λ = 3.0;                    # discrete jump duration (Poisson rate)
MAX_ITER = 60;              # Baum-Welch max EM iterations
N_PATHS = 1000;             # Monte Carlo simulation paths
L = 252;                    # ACF max lag (1 trading year)
risk_free_rate = 0.0;
Δt = 1/252;

## Load Data

In [ ]:
# --- LOAD IN-SAMPLE DATA ---
train_dataset = MyPortfolioDataSet() |> x -> x["dataset"];
maximum_number_trading_days = nrow(train_dataset["AAPL"]);

dataset = Dict{String,DataFrame}();
for (t, data) in train_dataset
    if nrow(data) == maximum_number_trading_days
        dataset[t] = data;
    end
end

list_of_all_tickers = keys(dataset) |> collect |> sort;
all_firms_R = log_growth_matrix(dataset, list_of_all_tickers; Δt=Δt, risk_free_rate=risk_free_rate);
ticker_idx = findfirst(x -> x == ticker, list_of_all_tickers);
R_is = all_firms_R[:, ticker_idx];
n_steps_is = length(R_is);

# --- LOAD OUT-OF-SAMPLE DATA ---
oos_dataset_raw = MyOutOfSamplePortfolioDataSet() |> x -> x["dataset"];
R_oos = log_growth_matrix(oos_dataset_raw, ticker; Δt=Δt, risk_free_rate=risk_free_rate);
n_steps_oos = length(R_oos);

println("IS data: $(n_steps_is) observations for $ticker");
println("OoS data: $(n_steps_oos) observations for $ticker");

## Evaluation Metrics Function

In [ ]:
function eval_metrics(observed::Vector{Float64}, simulated_archive::Matrix{Float64};
                      alpha=0.05, L_acf=252)

    n_paths = size(simulated_archive, 2);
    n_obs = length(observed);

    # Observed targets
    mu_obs = mean(observed); sigma_obs = std(observed);
    kurt_obs = sum(((observed .- mu_obs) ./ sigma_obs).^4) / n_obs - 3.0;
    tau_range = 1:min(L_acf, n_obs - 1);
    acf_obs = autocor(abs.(observed), tau_range);

    # Accumulators
    ks_pass = 0;
    kurt_sum = 0.0;
    acf_mae_sum = 0.0;
    w1_sum = 0.0;
    hell_sum = 0.0;
    ks_pvals = Float64[];

    for i in 1:n_paths
        sim = simulated_archive[:, i];

        # KS test
        pval = pvalue(ApproximateTwoSampleKSTest(observed, sim));
        push!(ks_pvals, pval);
        if pval > alpha; ks_pass += 1; end

        # Kurtosis
        mu_s = mean(sim); sigma_s = std(sim);
        kurt_sum += sum(((sim .- mu_s) ./ sigma_s).^4) / length(sim) - 3.0;

        # ACF-MAE on absolute returns
        acf_sim = autocor(abs.(sim), tau_range);
        acf_mae_sum += mean(abs.(acf_obs .- acf_sim));

        # Wasserstein-1 (sorted quantile difference)
        obs_sorted = sort(observed);
        sim_sorted = sort(sim);
        n_min = min(length(obs_sorted), length(sim_sorted));
        obs_q = [obs_sorted[max(1, round(Int, k * length(obs_sorted) / n_min))] for k in 1:n_min];
        sim_q = [sim_sorted[max(1, round(Int, k * length(sim_sorted) / n_min))] for k in 1:n_min];
        w1_sum += mean(abs.(obs_q .- sim_q));

        # Hellinger distance (histogram-based)
        lo = min(minimum(observed), minimum(sim)) - 10;
        hi = max(maximum(observed), maximum(sim)) + 10;
        edges = range(lo, hi, length=101);
        h_obs = fit(Histogram, observed, edges).weights ./ n_obs;
        h_sim = fit(Histogram, sim, edges).weights ./ length(sim);
        hell_sum += sqrt(sum((sqrt.(h_obs) .- sqrt.(h_sim)).^2)) / sqrt(2);
    end

    return (
        ks_rate     = round(100.0 * ks_pass / n_paths, digits=1),
        kurtosis_obs = round(kurt_obs, digits=2),
        kurtosis_sim = round(kurt_sum / n_paths, digits=2),
        acf_mae     = round(acf_mae_sum / n_paths, digits=4),
        wasserstein = round(w1_sum / n_paths, digits=3),
        hellinger   = round(hell_sum / n_paths, digits=4),
        ks_pvals    = ks_pvals
    );
end;

## Fit All 3 Models

### Model 1: Discrete HMM + Jumps

Discretize returns into `K_discrete` quantile bins, count transition frequencies,
and build the jump model with Poisson regime teleportation.

In [ ]:
# --- STEP 1: Discretize returns into K_discrete quantile bins ---
states_d = collect(1:K_discrete);

# Compute quantile cutoffs from the empirical CDF
d_empirical = fit(Normal, R_is);   # Gaussian fit for quantile boundaries
percentage_cutoff = range(0.0, stop=1.0, length=(K_discrete + 1)) |> collect;

# Compute bin bounds from quantiles
bounds = Array{Float64,2}(undef, K_discrete, 3);
for s in states_d
    bounds[s, 1] = quantile(d_empirical, percentage_cutoff[s]);
    bounds[s, 2] = quantile(d_empirical, percentage_cutoff[s + 1]);
    bounds[s, 3] = s;
end

# Encode: assign each observation to a bin
encoded_is = Array{Int64,1}();
for i in eachindex(R_is)
    value = R_is[i];
    class_index = 1;
    for s in states_d
        if (s == K_discrete)
            class_index = s;
            break;
        elseif (value >= bounds[s, 1] && value < bounds[s, 2])
            class_index = s;
            break;
        end
    end
    push!(encoded_is, class_index);
end

# Build transition matrix by frequency counting
T_count = zeros(K_discrete, K_discrete);
for i in 2:length(encoded_is)
    T_count[encoded_is[i-1], encoded_is[i]] += 1;
end

T_discrete = zeros(K_discrete, K_discrete);
for row in states_d
    Z = sum(T_count[row, :]);
    if Z > 0
        T_discrete[row, :] = T_count[row, :] ./ Z;
    else
        T_discrete[row, :] .= 1.0 / K_discrete;
    end
end

# Identity emission matrix (deterministic emissions)
E_discrete = diagm(ones(K_discrete));

# Build decode distributions: Normal fit per bin from observed data
decode_distribution = Dict{Int64, Normal}();
for s in states_d
    idx_s = findall(x -> x == s, encoded_is);
    if length(idx_s) > 1
        decode_distribution[s] = fit(Normal, R_is[idx_s]);
    else
        # Fallback: use bin midpoint with small std
        mid = (bounds[s, 1] + bounds[s, 2]) / 2.0;
        decode_distribution[s] = Normal(mid, 1.0);
    end
end

# Build jump model
discrete_jump_model = build(MyHiddenMarkovModelWithJumps, (
    states = states_d,
    T = T_discrete,
    E = E_discrete,
    ϵ = ε,
    λ = λ
));

# Stationary distribution for discrete model
pi_discrete = Categorical((T_discrete^1000)[1, :]);

println("Discrete HMM + Jumps: K=$(K_discrete), epsilon=$(ε), lambda=$(λ)");
println("  Transition matrix: $(K_discrete) x $(K_discrete)");
println("  Decode distributions built for all $(K_discrete) bins");

### Model 2: Continuous HMM (Baum-Welch)

In [ ]:
# --- TRAIN CHMM ---
chmm_model = build(MyContinuousHiddenMarkovModel, (
    observations = R_is,
    number_of_states = K_continuous,
    max_iter = MAX_ITER
));

# Stationary distribution
T_chmm = zeros(K_continuous, K_continuous);
for i in 1:K_continuous
    T_chmm[i, :] = probs(chmm_model.transition[i]);
end
pi_chmm = Categorical((T_chmm^1000)[1, :]);

println("Continuous HMM (Baum-Welch): K=$(K_continuous)");
println("  Converged in $(length(chmm_model.log_likelihood_history)) iterations");
println("  Emission means: ", [round(mean(chmm_model.emission[s]), digits=1) for s in 1:K_continuous]);
println("  Emission stds:  ", [round(std(chmm_model.emission[s]), digits=1) for s in 1:K_continuous]);

### Model 3: GARCH(1,1)

In [ ]:
# --- FIT GARCH(1,1) ---
garch_model = build(MyGARCHModel, (
    observations = R_is,
));

println("GARCH(1,1) fitted:");
println("  omega = $(round(garch_model.ω, digits=4))");
println("  alpha = $(round(garch_model.α, digits=4))");
println("  beta  = $(round(garch_model.β, digits=4))");
println("  mu    = $(round(garch_model.μ, digits=4))");
println("  persistence (alpha+beta) = $(round(garch_model.α + garch_model.β, digits=4))");
println("  log-likelihood = $(round(garch_model.log_likelihood, digits=2))");

## Simulate from All 3 Models (IS Length)

In [ ]:
# --- SIMULATE: DISCRETE + JUMPS ---
println("Simulating $(N_PATHS) paths from Discrete+Jumps...");
sim_discrete = Array{Float64,2}(undef, n_steps_is, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(pi_discrete);
    state_chain = discrete_jump_model(s0, n_steps_is);
    for j in 1:n_steps_is
        sim_discrete[j, i] = rand(decode_distribution[state_chain[j]]);
    end
end

# --- SIMULATE: CHMM ---
println("Simulating $(N_PATHS) paths from CHMM...");
sim_chmm = Array{Float64,2}(undef, n_steps_is, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(pi_chmm);
    state_chain = chmm_model(s0, n_steps_is);
    for j in 1:n_steps_is
        sim_chmm[j, i] = rand(chmm_model.emission[state_chain[j]]);
    end
end

# --- SIMULATE: GARCH ---
println("Simulating $(N_PATHS) paths from GARCH(1,1)...");
sim_garch = Array{Float64,2}(undef, n_steps_is, N_PATHS);
for i in 1:N_PATHS
    sim_garch[:, i] = simulate_garch(garch_model, n_steps_is);
end

println("All simulations complete: $(N_PATHS) paths x $(n_steps_is) steps each.");

## Compute In-Sample Metrics for All 3 Models

In [ ]:
println("Computing IS metrics for Discrete+Jumps...");
m_discrete_is = eval_metrics(R_is, sim_discrete; L_acf=L);

println("Computing IS metrics for CHMM...");
m_chmm_is = eval_metrics(R_is, sim_chmm; L_acf=L);

println("Computing IS metrics for GARCH...");
m_garch_is = eval_metrics(R_is, sim_garch; L_acf=L);

println("IS metrics computed.");

## Head-to-Head Comparison Table (In-Sample)

In [ ]:
println("\n" * "="^85);
println("  IN-SAMPLE MODEL COMPARISON --- $(ticker)");
println("  $(N_PATHS) simulated paths, $(n_steps_is) steps each");
println("="^85);
println(
    rpad("Metric", 22) *
    rpad("Observed", 14) *
    rpad("Disc+Jumps", 14) *
    rpad("CHMM (BW)", 14) *
    rpad("GARCH(1,1)", 14)
);
println("-"^85);
println(
    rpad("KS pass rate (%)", 22) *
    rpad("---", 14) *
    rpad(m_discrete_is.ks_rate, 14) *
    rpad(m_chmm_is.ks_rate, 14) *
    rpad(m_garch_is.ks_rate, 14)
);
println(
    rpad("Excess kurtosis", 22) *
    rpad(m_discrete_is.kurtosis_obs, 14) *
    rpad(m_discrete_is.kurtosis_sim, 14) *
    rpad(m_chmm_is.kurtosis_sim, 14) *
    rpad(m_garch_is.kurtosis_sim, 14)
);
println(
    rpad("ACF-MAE |Gt|", 22) *
    rpad("---", 14) *
    rpad(m_discrete_is.acf_mae, 14) *
    rpad(m_chmm_is.acf_mae, 14) *
    rpad(m_garch_is.acf_mae, 14)
);
println(
    rpad("Wasserstein-1", 22) *
    rpad("---", 14) *
    rpad(m_discrete_is.wasserstein, 14) *
    rpad(m_chmm_is.wasserstein, 14) *
    rpad(m_garch_is.wasserstein, 14)
);
println(
    rpad("Hellinger", 22) *
    rpad("---", 14) *
    rpad(m_discrete_is.hellinger, 14) *
    rpad(m_chmm_is.hellinger, 14) *
    rpad(m_garch_is.hellinger, 14)
);
println("="^85);

## Figure: Density Comparison (Single Panel)

Observed distribution overlaid with kernel density estimates from all 3 models.

In [ ]:
# --- DENSITY COMPARISON ---
x_lo_raw = quantile(R_is, 0.005); x_hi_raw = quantile(R_is, 0.995);
x_pad = 0.20 * (x_hi_raw - x_lo_raw);
x_lo = x_lo_raw - x_pad; x_hi = x_hi_raw + x_pad;

p_density = plot(title="Marginal Density Comparison --- $(ticker) (IS)",
    titlefontsize=10, xlabel="Excess Growth Rate", ylabel="Probability Density (AU)",
    legend=:topright, size=(800, 500));

histogram!(p_density, R_is, normalize=:pdf, bins=200, alpha=0.3, color=:lightgray, label="Observed");
density!(p_density, vec(sim_discrete), lw=2, color=:green, alpha=0.8, label="Discrete+Jumps (K=$(K_discrete))");
density!(p_density, vec(sim_chmm), lw=2, color=:blue, alpha=0.8, label="CHMM (K=$(K_continuous))");
density!(p_density, vec(sim_garch), lw=2, color=:red, alpha=0.8, ls=:dash, label="GARCH(1,1)");
xlims!(p_density, x_lo, x_hi);

display(p_density);

## Figure: ACF(|Gt|) Comparison (Single Panel)

Mean autocorrelation of absolute returns from each model vs observed.

In [ ]:
# --- ACF(|Gt|) COMPARISON ---
tau = 1:L;
acf_obs = autocor(abs.(R_is), tau);

n_acf_sample = min(200, N_PATHS);

# Discrete+Jumps mean ACF
acf_discrete_arch = hcat([autocor(abs.(sim_discrete[:, i]), tau) for i in 1:n_acf_sample]...);
acf_discrete_mean = mean(acf_discrete_arch, dims=2)[:];

# CHMM mean ACF
acf_chmm_arch = hcat([autocor(abs.(sim_chmm[:, i]), tau) for i in 1:n_acf_sample]...);
acf_chmm_mean = mean(acf_chmm_arch, dims=2)[:];

# GARCH mean ACF
acf_garch_arch = hcat([autocor(abs.(sim_garch[:, i]), tau) for i in 1:n_acf_sample]...);
acf_garch_mean = mean(acf_garch_arch, dims=2)[:];

p_acf = plot(tau, acf_obs, lw=2.5, color=:black, label="Observed",
    title="ACF(|Gt|) Comparison --- $(ticker) (IS)", titlefontsize=10,
    xlabel="Lag (trading days)", ylabel="ACF(|Gt|)",
    legend=:topright, size=(800, 500));
plot!(p_acf, tau, acf_discrete_mean, lw=2, color=:green, label="Discrete+Jumps");
plot!(p_acf, tau, acf_chmm_mean, lw=2, color=:blue, label="CHMM");
plot!(p_acf, tau, acf_garch_mean, lw=2, color=:red, ls=:dash, label="GARCH(1,1)");

display(p_acf);

## Figure: Q-Q Comparison (Single Panel)

Quantile-quantile plot (0.1st to 99.9th percentile) for each model vs observed.

In [ ]:
# --- Q-Q COMPARISON ---
probs_qq = range(0.001, 0.999, length=200);
q_obs = quantile(R_is, probs_qq);
q_discrete = quantile(vec(sim_discrete), probs_qq);
q_chmm = quantile(vec(sim_chmm), probs_qq);
q_garch = quantile(vec(sim_garch), probs_qq);

p_qq = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="Q-Q Comparison (0.1st-99.9th pctl) --- $(ticker) (IS)", titlefontsize=10,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles",
    legend=:topleft, size=(700, 600));
scatter!(p_qq, q_obs, q_discrete, ms=3, alpha=0.6, color=:green, label="Discrete+Jumps");
scatter!(p_qq, q_obs, q_chmm, ms=3, alpha=0.6, color=:blue, label="CHMM");
scatter!(p_qq, q_obs, q_garch, ms=3, alpha=0.6, color=:red, markershape=:diamond, label="GARCH(1,1)");

display(p_qq);

## Figure: Combined 3-Panel Comparison

Save the density, ACF, and Q-Q panels as a single figure.

In [ ]:
# --- REBUILD PANELS FOR COMBINED FIGURE ---

# Panel (a): Density
pa = plot(title="(a) Marginal Density", titlefontsize=9,
    xlabel="Excess Growth Rate", ylabel="Density (AU)");
histogram!(pa, R_is, normalize=:pdf, bins=200, alpha=0.3, color=:lightgray, label="Observed");
density!(pa, vec(sim_discrete), lw=2, color=:green, alpha=0.8, label="Disc+J");
density!(pa, vec(sim_chmm), lw=2, color=:blue, alpha=0.8, label="CHMM");
density!(pa, vec(sim_garch), lw=2, color=:red, ls=:dash, alpha=0.8, label="GARCH");
xlims!(pa, x_lo, x_hi);

# Panel (b): ACF
pb = plot(tau, acf_obs, lw=2.5, color=:black, label="Observed",
    title="(b) ACF(|Gt|)", titlefontsize=9,
    xlabel="Lag", ylabel="ACF(|Gt|)");
plot!(pb, tau, acf_discrete_mean, lw=2, color=:green, label="Disc+J");
plot!(pb, tau, acf_chmm_mean, lw=2, color=:blue, label="CHMM");
plot!(pb, tau, acf_garch_mean, lw=2, color=:red, ls=:dash, label="GARCH");

# Panel (c): Q-Q
pc = plot(q_obs, q_obs, lw=2, color=:black, ls=:dash, label="Perfect",
    title="(c) Q-Q (0.1st-99.9th)", titlefontsize=9,
    xlabel="Observed Quantiles", ylabel="Simulated Quantiles");
scatter!(pc, q_obs, q_discrete, ms=2, alpha=0.6, color=:green, label="Disc+J");
scatter!(pc, q_obs, q_chmm, ms=2, alpha=0.6, color=:blue, label="CHMM");
scatter!(pc, q_obs, q_garch, ms=2, alpha=0.6, color=:red, markershape=:diamond, label="GARCH");

# Combine and save
fig_combined = plot(pa, pb, pc, layout=(1, 3), size=(1500, 450),
    plot_title="Model Comparison --- $(ticker) (IS, $(N_PATHS) paths)",
    plot_titlefontsize=12);
display(fig_combined);

savefig(fig_combined, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-Model-Comparison-IS.svg"));
savefig(fig_combined, joinpath(_PATH_TO_FIGURES, "Fig-$(ticker)-Model-Comparison-IS.pdf"));
println("Saved: figs/Fig-$(ticker)-Model-Comparison-IS.svg");

## Out-of-Sample Comparison

Simulate at OoS length, compute metrics, and print the comparison table.

In [ ]:
# --- SIMULATE OoS-LENGTH PATHS ---
println("Simulating OoS-length paths ($(n_steps_oos) steps) from all 3 models...");

# Discrete+Jumps OoS
sim_discrete_oos = Array{Float64,2}(undef, n_steps_oos, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(pi_discrete);
    state_chain = discrete_jump_model(s0, n_steps_oos);
    for j in 1:n_steps_oos
        sim_discrete_oos[j, i] = rand(decode_distribution[state_chain[j]]);
    end
end

# CHMM OoS
sim_chmm_oos = Array{Float64,2}(undef, n_steps_oos, N_PATHS);
for i in 1:N_PATHS
    s0 = rand(pi_chmm);
    state_chain = chmm_model(s0, n_steps_oos);
    for j in 1:n_steps_oos
        sim_chmm_oos[j, i] = rand(chmm_model.emission[state_chain[j]]);
    end
end

# GARCH OoS
sim_garch_oos = Array{Float64,2}(undef, n_steps_oos, N_PATHS);
for i in 1:N_PATHS
    sim_garch_oos[:, i] = simulate_garch(garch_model, n_steps_oos);
end

println("OoS simulation complete.");

In [ ]:
# --- COMPUTE OoS METRICS ---
println("Computing OoS metrics...");
m_discrete_oos = eval_metrics(R_oos, sim_discrete_oos; L_acf=L);
m_chmm_oos = eval_metrics(R_oos, sim_chmm_oos; L_acf=L);
m_garch_oos = eval_metrics(R_oos, sim_garch_oos; L_acf=L);

println("\n" * "="^85);
println("  OUT-OF-SAMPLE MODEL COMPARISON --- $(ticker)");
println("  $(N_PATHS) simulated paths, $(n_steps_oos) steps each");
println("="^85);
println(
    rpad("Metric", 22) *
    rpad("Observed", 14) *
    rpad("Disc+Jumps", 14) *
    rpad("CHMM (BW)", 14) *
    rpad("GARCH(1,1)", 14)
);
println("-"^85);
println(
    rpad("KS pass rate (%)", 22) *
    rpad("---", 14) *
    rpad(m_discrete_oos.ks_rate, 14) *
    rpad(m_chmm_oos.ks_rate, 14) *
    rpad(m_garch_oos.ks_rate, 14)
);
println(
    rpad("Excess kurtosis", 22) *
    rpad(m_discrete_oos.kurtosis_obs, 14) *
    rpad(m_discrete_oos.kurtosis_sim, 14) *
    rpad(m_chmm_oos.kurtosis_sim, 14) *
    rpad(m_garch_oos.kurtosis_sim, 14)
);
println(
    rpad("ACF-MAE |Gt|", 22) *
    rpad("---", 14) *
    rpad(m_discrete_oos.acf_mae, 14) *
    rpad(m_chmm_oos.acf_mae, 14) *
    rpad(m_garch_oos.acf_mae, 14)
);
println(
    rpad("Wasserstein-1", 22) *
    rpad("---", 14) *
    rpad(m_discrete_oos.wasserstein, 14) *
    rpad(m_chmm_oos.wasserstein, 14) *
    rpad(m_garch_oos.wasserstein, 14)
);
println(
    rpad("Hellinger", 22) *
    rpad("---", 14) *
    rpad(m_discrete_oos.hellinger, 14) *
    rpad(m_chmm_oos.hellinger, 14) *
    rpad(m_garch_oos.hellinger, 14)
);
println("="^85);

## State Sensitivity (Optional)

How does the CHMM's performance change as K varies? We try K in {3, 6, 9}
and compare the key metrics.

In [ ]:
# --- CHMM STATE SENSITIVITY ---
K_sweep = [3, 6, 9];
sensitivity_results = Dict{Int, NamedTuple}();

for K_val in K_sweep
    println("Training CHMM with K=$(K_val)...");

    model_k = build(MyContinuousHiddenMarkovModel, (
        observations = R_is,
        number_of_states = K_val,
        max_iter = MAX_ITER
    ));

    # Stationary distribution
    T_k = zeros(K_val, K_val);
    for i in 1:K_val; T_k[i, :] = probs(model_k.transition[i]); end
    pi_k = Categorical((T_k^1000)[1, :]);

    # Simulate
    sim_k = Array{Float64,2}(undef, n_steps_is, N_PATHS);
    for p in 1:N_PATHS
        s0 = rand(pi_k);
        states_chain = model_k(s0, n_steps_is);
        for j in 1:n_steps_is
            sim_k[j, p] = rand(model_k.emission[states_chain[j]]);
        end
    end

    # Evaluate
    m_k = eval_metrics(R_is, sim_k; L_acf=L);
    sensitivity_results[K_val] = m_k;

    println("  K=$(K_val): KS=$(m_k.ks_rate)%, ACF-MAE=$(m_k.acf_mae), Kurt=$(m_k.kurtosis_sim) (obs=$(m_k.kurtosis_obs))");
end

In [ ]:
# --- SENSITIVITY TABLE ---
println("\n" * "="^75);
println("  CHMM STATE SENSITIVITY --- $(ticker)");
println("="^75);
println(
    rpad("K", 6) *
    rpad("KS (%)", 10) *
    rpad("Kurt (sim)", 14) *
    rpad("Kurt (obs)", 14) *
    rpad("ACF-MAE", 12) *
    rpad("W-1", 10) *
    rpad("Hellinger", 10)
);
println("-"^75);
for K_val in K_sweep
    m = sensitivity_results[K_val];
    println(
        rpad(K_val, 6) *
        rpad(m.ks_rate, 10) *
        rpad(m.kurtosis_sim, 14) *
        rpad(m.kurtosis_obs, 14) *
        rpad(m.acf_mae, 12) *
        rpad(m.wasserstein, 10) *
        rpad(m.hellinger, 10)
    );
end
println("="^75);
println("Paths: $(N_PATHS) | MAX_ITER: $(MAX_ITER)");

## Disclaimer

This content is offered solely for research and educational purposes.
It does not constitute financial advice. The models presented here are
academic tools for studying stylized facts of financial returns and
should not be used for trading decisions.